In [116]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

pd.set_option('display.max_colwidth', None)  # Display full content of each column
pd.set_option('display.max_columns', None)   # Display all columns
pd.set_option('display.width', 5000)         # Set display width

MERGE CSV 
---

In [117]:
# import msoffcrypto
# import io

# password = "YASH27032004"

# files = ["2024.xlsx", "2025.xlsx", "2026.xlsx"]
# dfs = []

# for file_name in files:
#     decrypted = io.BytesIO()

#     with open(file_name, "rb") as f:
#         office_file = msoffcrypto.OfficeFile(f)
#         office_file.load_key(password=password)
#         office_file.decrypt(decrypted)

#     df = pd.read_excel(decrypted, skiprows=17)
#     dfs.append(df)

# final_df = pd.concat(dfs, ignore_index=True)
# final_df.to_excel("YashSawant_3yrs.xlsx", index=False)


REGEX
--

In [118]:
import re
import pandas as pd

# List of all transaction modes
transaction_modes = [
    "UPI", "INB", "IMP", "NEFT", "RTGS", "Cheque", "Cash Deposit", "Cash Withdrawal",
    "POS", "DD", "SWIFT", "Wire Transfer", "ECS", "Bill Pay", "M-wallet", "EMI", "EFT", "ACH"
]

pattern = re.compile(
    r"(?P<type>(?:WDL|DEP)\s+TFR)\s+"
    r"(?P<mode>[A-Z]+)/(?P<drcr>DR|CR)/"
    r"(?P<id>\d+)/"
    r"(?P<name>[^/]+)/"
    r"(?P<bank>[A-Z]+)/"
    r"(?P<upi_id>[^\s/]+)"
    r"(?:\s*-?\d+/(?P<note>[A-Za-z]+))?"
)

def extract_details(description):
    
    description = description.replace("\n", " ").strip()

    match = pattern.search(description)

    if match:
        return pd.Series({
            "Transaction_Type": match.group("type"),
            "Transaction_Mode": match.group("mode"),
            "DR/CR_Indicator": match.group("drcr"),
            "Transaction_ID": match.group("id"),
            "Recipient_Name": match.group("name"),
            "Bank": match.group("bank"),
            "UPI_ID": match.group("upi_id"),
            "Note": match.group("note") if match.group("note") else "N/A"
        })

    else:
        description_clean = description.replace("\n", " ").strip()
        split_data = description_clean.split('/')

        # Extract transaction mode
        transaction_mode = "N/A"
        for mode in transaction_modes:
            if mode in description_clean:
                transaction_mode = mode
                break

        # Extract note
        note_match = re.search(r'/\s*([A-Za-z]+)\s+\d+', description_clean)
        note = note_match.group(1) if note_match else "N/A"

        txn_id_match = re.search(r'\d{6,}', split_data[0])
        txn_id = txn_id_match.group() if txn_id_match else "N/A"

        return pd.Series({
            "Transaction_Type": "DEP TFR" if "DEP TFR" in description_clean else "N/A",
            "Transaction_Mode": transaction_mode,
            "DR/CR_Indicator": "CR" if "DEP" in description_clean else "N/A",
            "Transaction_ID": txn_id,
            "Recipient_Name": split_data[1] if len(split_data) > 1 else "N/A",
            "Bank": "N/A",
            "UPI_ID": "N/A",
            "Note": note
        })
# text = """ DEP TFR   UPI/CR/345946067401/SNEHA DI/HDFC/ss280387
#  -2/UPI   0093009162091 AT 71097 EVERSHINE CITY BRANCH"""
# print(extract_details(text))

Rename Columns
---

In [119]:
df = pd.read_excel("YashSawant_3yrs.xlsx")

df = df.replace('\n', '', regex=True)

df = df.rename(columns={
    'Date': 'Transaction_Date',    
    'Details': 'Description',
    'Ref No./Cheque\nNo.': 'Reference No./Cheque No.',
    'Debit': 'Debit',
    'Credit': 'Credit',
    'Balance': 'Balance'
})
df_extracted = df['Description'].apply(extract_details)

df = pd.concat([df, df_extracted], axis=1)
df


,Transaction_Date,Description,Ref No/Cheque No,Debit,Credit,Balance,Transaction_Type,Transaction_Mode,DR/CR_Indicator,Transaction_ID,Recipient_Name,Bank,UPI_ID,Note
0,01/04/2023,DEP TFR INB IMPS309111290781/9890160567/XX8237/ Son 0098030162099 AT 71097 EVERSHINE CITY BRANCH,NaN,NaN,1500.0,1614.48,DEP TFR,INB,CR,309111290781,9890160567,N/A,N/A,Son
1,01/04/2023,WDL TFR UPI/DR/309122218462/ASIM HEM/HDFC/asimshah 21/UPI 0099744162096 AT 71097 EVERSHINE CITY BRANCH,NaN,75.0,NaN,1539.48,WDL TFR,UPI,DR,309122218462,ASIM HEM,HDFC,asimshah,UPI
2,02/04/2023,WDL TFR UPI/DR/309252494278/SAI RAJ /PYTM/paytmqr2 81/UPI 0096407162097 AT 71097 EVERSHINE CITY BRANCH,NaN,1102.0,NaN,437.48,WDL TFR,UPI,DR,309252494278,SAI RAJ,PYTM,paytmqr2,UPI
3,02/04/2023,WDL TFR UPI/DR/309255302589/PRACHI S/SBIN/prachi24 77/UPI 0099846162090 AT 71097 EVERSHINE CITY BRANCH,NaN,10.0,NaN,427.48,WDL TFR,UPI,DR,309255302589,PRACHI S,SBIN,prachi24,UPI
4,02/04/2023,DEP TFR UPI/CR/309220310152/PRACHI S/SBIN/prachisw t2/Pay 0098828162099 AT 71097 EVERSHINE CITY BRANCH,NaN,NaN,1500.0,1927.48,DEP TFR,UPI,CR,309220310152,PRACHI S,SBIN,prachisw,N/A
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1648,13/03/2026,WDL TFR UPI/DR/600865604383/ASLAM HA/PUNB/ak3967270 @/Sent 0097694162092 AT 71097 EVERSHINE CITY BRANCH,NaN,58.0,NaN,1631.07,WDL TFR,UPI,DR,600865604383,ASLAM HA,PUNB,ak3967270,N/A
1649,13/03/2026,WDL TFR UPI/DR/607214745446/KEWAL BI/HDFC/kewal210 50/UPI 0097694162092 AT 71097 EVERSHINE CITY BRANCH,NaN,125.0,NaN,1506.07,WDL TFR,UPI,DR,607214745446,KEWAL BI,HDFC,kewal210,UPI
1650,13/03/2026,WDL TFR UPI/DR/607214758457/MIHIR YO/ICIC/mihirheb al/UPI 0097694162092 AT 71097 EVERSHINE CITY BRANCH,NaN,275.0,NaN,1231.07,WDL TFR,UPI,DR,607214758457,MIHIR YO,ICIC,mihirheb,N/A
1651,13/03/2026,WDL TFR UPI/DR/607289191772/Indian R/SBIN/railsbiup i/Sent 0097694162092 AT 71097 EVERSHINE CITY BRANCH,NaN,590.0,NaN,641.07,WDL TFR,UPI,DR,607289191772,Indian R,SBIN,railsbiup,N/A


Drop Columns
---

In [120]:
df = df.drop(columns=['Ref No/Cheque No', 'Transaction_Type', 'Description'])

Formatting
---

In [121]:
# Convert to datetime
df['Transaction_Date'] = pd.to_datetime(df['Transaction_Date'], format='mixed', dayfirst=True)
df['Transaction_Date'] = df['Transaction_Date'].dt.date.astype(str)

# Numeric cleaning
df['Debit'] = pd.to_numeric(df['Debit'].replace({',': ''}, regex=True), errors='coerce')
df['Credit'] = pd.to_numeric(df['Credit'].replace({',': ''}, regex=True), errors='coerce')
df['Balance'] = pd.to_numeric(df['Balance'].replace({',': ''}, regex=True), errors='coerce')

# Fill debit/credit
df['Debit'] = df['Debit'].fillna(0)
df['Credit'] = df['Credit'].fillna(0)

# DR/CR logic
def determine_dr_cr(df):
    if df.loc[0, 'Credit'] > 0:
        df.loc[0, 'DR/CR_Indicator'] = 'CR'
    elif df.loc[0, 'Debit'] > 0:
        df.loc[0, 'DR/CR_Indicator'] = 'DR'
    else:
        df.loc[0, 'DR/CR_Indicator'] = None
    
    for i in range(1, len(df)):
        balance_diff = df.loc[i, 'Balance'] - df.loc[i-1, 'Balance']
        df.loc[i, 'DR/CR_Indicator'] = 'CR' if balance_diff > 0 else 'DR'
    
    return df

df = determine_dr_cr(df)

# Amount
df['Amount'] = df['Credit'] - df['Debit']

In [122]:
print(df.isna().sum())


Transaction_Date    0
Debit               0
Credit              0
Balance             0
Transaction_Mode    0
DR/CR_Indicator     0
Transaction_ID      0
Recipient_Name      0
Bank                0
UPI_ID              0
Note                0
Amount              0
dtype: int64


In [123]:
for i in df.columns:
    print(i)
# df

Transaction_Date
Debit
Credit
Balance
Transaction_Mode
DR/CR_Indicator
Transaction_ID
Recipient_Name
Bank
UPI_ID
Note
Amount


In [130]:
import pandas as pd
import numpy as np
import math

def clean_recipient(name):
    if pd.isna(name):
        return "UNKNOWN"
    name = str(name).strip()
    if name.isdigit():
        return "PHONE_TRANSFER"
    return name

df["Recipient_Name"] = df["Recipient_Name"].apply(clean_recipient)

df.replace("N/A", None, inplace=True)
df = df.replace([np.inf, -np.inf], None)
df = df.astype(object)
df = df.where(pd.notna(df), None)

records = df.to_dict(orient="records")

def clean_record(r):
    clean = {}
    for k, v in r.items():
        if v is None:
            clean[k] = None
        elif isinstance(v, float) and (math.isnan(v) or math.isinf(v)):
            clean[k] = None
        elif str(v).lower() == 'nan':   # 🔥 IMPORTANT FIX
            clean[k] = None
        else:
            clean[k] = v
    return clean

records = [clean_record(r) for r in records]

for r in records:
    for v in r.values():
        if isinstance(v, float) and math.isnan(v):
            print("❌ FLOAT NaN FOUND:", r)
        if str(v).lower() == 'nan':
            print("❌ STRING NaN FOUND:", r)

print("✅ 100% CLEAN DATA - READY FOR INSERT")

✅ 100% CLEAN DATA - READY FOR INSERT


Export
---

In [132]:
df.to_excel("SpendWise2k26.xlsx", index=False)

In [126]:
import math
def validate_dataframe(df):
    errors = []

    for idx, row in df.iterrows():
        row_errors = []

        for col, val in row.items():

            # Check None (will become null in JSON)
            if val is None:
                continue  # allowed

            # Check invalid floats (extra safety)
            if isinstance(val, float):
                if math.isnan(val) or math.isinf(val):
                    row_errors.append(f"{col} = Invalid float ({val})")

            # Check empty strings
            if isinstance(val, str) and val.strip() == "":
                row_errors.append(f"{col} = Empty string")

        if row_errors:
            errors.append({
                "row_index": idx,
                "errors": row_errors,
                "row_data": row.to_dict()
            })

    return errors

# -----------------------------
# Run Validation
# -----------------------------
errors = validate_dataframe(df)

# -----------------------------
# Result
# -----------------------------
if not errors:
    print("✅ Data is CLEAN and ready for Supabase insert!")

    # Convert to records (ready for insert)
    records = df.to_dict(orient="records")
    print(f"Total records ready: {len(records)}")

else:
    print(f"❌ Found {len(errors)} problematic rows\n")

    for e in errors[:5]:  # show first 5 errors
        print(f"Row {e['row_index']}:")
        for err in e["errors"]:
            print("  -", err)
        print()

    # Save bad rows for debugging
    bad_rows = [e["row_data"] for e in errors]
    pd.DataFrame(bad_rows).to_excel("bad_data.xlsx", index=False)

    print("⚠️ Bad rows exported to 'bad_data.xlsx'")

✅ Data is CLEAN and ready for Supabase insert!
Total records ready: 1653


In [131]:
print(df.isna().sum())

Transaction_Date      0
Debit                 0
Credit                0
Balance               0
Transaction_Mode     36
DR/CR_Indicator       0
Transaction_ID      101
Recipient_Name        0
Bank                180
UPI_ID              180
Note                914
Amount                0
dtype: int64


In [128]:
na_count_per_column = df.applymap(lambda x: x == "N/A").sum()

# Print the result
print("Number of 'N/A' values per column:")
print(na_count_per_column)

Number of 'N/A' values per column:
Transaction_Date    0
Debit               0
Credit              0
Balance             0
Transaction_Mode    0
DR/CR_Indicator     0
Transaction_ID      0
Recipient_Name      0
Bank                0
UPI_ID              0
Note                0
Amount              0
dtype: int64


In [129]:
transaction_modes = [
    "UPI",               # Unified Payments Interface
    "INB",               # Internet Banking
    "IMP",               # Immediate Payment Service
    "NEFT",              # National Electronic Funds Transfer
    "RTGS",              # Real-Time Gross Settlement
    "Cheque",            # Paper Cheque
    "Cash Deposit",      # Cash Deposit to account
    "Cash Withdrawal",   # Withdrawal using cash
    "POS",               # Point of Sale (Card Payment)
    "DD",                # Demand Draft
    "SWIFT",             # Society for Worldwide Interbank Financial Telecommunication
    "Wire Transfer",     # Electronic bank transfer
    "ECS",               # Electronic Clearing Service
    "Bill Pay",          # Utility Bill Payments
    "M-wallet",          # Mobile Wallet Transfer
    "EMI",               # Equated Monthly Installment
    "EFT",               # Electronic Funds Transfer
    "ACH",               # Automated Clearing House
]
